In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.data import load_region_df
from src.models import train_model
from src.forecaster import (
    HOUSTON_SPREADS,
    forecast_all_spreads,
    build_forecast_dispatch_schedule,
    forecast_error_cost,
    forecast_accuracy,
)

passed = 0
failed = 0

def check(condition, msg_pass, msg_fail):
    global passed, failed
    if condition:
        print(f"✅ {msg_pass}")
        passed += 1
    else:
        print(f"❌ {msg_fail}")
        failed += 1

# ── Setup ──────────────────────────────────────────────────────────────────
df_coast = load_region_df("coast")
region_dfs = {"coast": df_coast}

# Train one model per Houston spread
print("Training models for all 4 Houston spreads...")
zone_models = {}
for spread_col in HOUSTON_SPREADS.keys():
    if spread_col in df_coast.columns:
        zone_models[spread_col] = train_model(df_coast, spread_col)
        print(f"  ✅ {spread_col}: MAE=${zone_models[spread_col]['mae']:.2f}")

start = pd.Timestamp("2025-06-01")
end   = pd.Timestamp("2025-06-07")

# ── Test A — forecast_all_spreads returns correct structure ───────────────
forecast_df = forecast_all_spreads(zone_models, region_dfs, start, end)

check(len(forecast_df) == 7 * 24,
      f"Test A — {7*24} hourly rows returned",
      f"Test A — wrong row count: {len(forecast_df)}")

for sc in HOUSTON_SPREADS.keys():
    check(f"{sc}_forecast" in forecast_df.columns,
          f"Test A — {sc}_forecast column present",
          f"Test A — {sc}_forecast column MISSING")
    check(f"{sc}_actual" in forecast_df.columns,
          f"Test A — {sc}_actual column present",
          f"Test A — {sc}_actual column MISSING")

# ── Test B — build_forecast_dispatch_schedule no simultaneous dispatch ─────
schedule = build_forecast_dispatch_schedule(
    forecast_df, battery_mw=100, duration_hours=4,
    charge_pct=25, discharge_pct=75, primary_spread="spread_h_s"
)
simultaneous = (
    (schedule["charge_mw"] > 0) & (schedule["discharge_mw"] > 0)
).sum()
check(simultaneous == 0,
      "Test B — no simultaneous charge/discharge",
      f"Test B — {simultaneous} simultaneous hours found")

check("forecast_spread" in schedule.columns,
      "Test B — forecast_spread column present",
      "Test B — forecast_spread column MISSING")

# ── Test C — forecast_error_cost arithmetic ───────────────────────────────
result = forecast_error_cost(100_000, 75_000)
check(result["forecast_error_cost"] == 25_000,
      "Test C — error cost = $25,000",
      f"Test C — wrong: ${result['forecast_error_cost']:,.0f}")
check(result["capture_ratio"] == 75.0,
      "Test C — capture ratio = 75%",
      f"Test C — wrong: {result['capture_ratio']}%")

# ── Test D — forecast_accuracy structure ──────────────────────────────────
acc_df = forecast_accuracy(forecast_df)
check(isinstance(acc_df, pd.DataFrame),
      "Test D — forecast_accuracy returns DataFrame",
      "Test D — wrong type")
check("Direction Accuracy (%)" in acc_df.columns,
      "Test D — Direction Accuracy column present",
      "Test D — Direction Accuracy column MISSING")
check(acc_df["Direction Accuracy (%)"].between(0, 100).all(),
      "Test D — all direction accuracies in 0–100%",
      "Test D — out of range values found")

print(f"\n  Accuracy summary:\n{acc_df.to_string(index=False)}\n")

# ── Test E — no Streamlit imports ─────────────────────────────────────────
with open("../src/forecaster.py") as f:
    source = f.read()
check("import streamlit" not in source,
      "Test E — no Streamlit imports in forecaster.py",
      "Test E — Streamlit import found")

print(f"\n{'─'*50}")
print(f"Results: {passed} passed / {failed} failed / {passed+failed} total")
print(f"{'─'*50}")